In [5]:
# Cell 1
# Mục đích: dựng kết nối BigQuery và các hàm dùng chung (đo MB, dry run, chạy thật có trần byte)
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
# client: kênh kết nối Python với BigQuery; location phải khớp vùng dataset (US)
client = bigquery.Client(project=PROJECT, location="US")

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi "đọc byte quá sớm" lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng byte sẽ quét, không chạy thật
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Se quet: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật: bỏ qua cache, đặt trần byte bị tính tiền (đổi MB sang byte)
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,
    )
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()  # result() chờ job xong rồi mới đọc byte
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

In [6]:
# Cell 2
# Mục đích: so sánh 3 nguồn: bảng incremental mới, fct_net_sales cũ, và view staging
# (lọc Complete) để biết vì sao bảng mới có 45,3k dòng chứ không phải 45,5k.
sql = """
SELECT 'inc' AS nguon, COUNT(*) AS so_dong, COUNT(DISTINCT order_item_id) AS so_id,
       COUNTIF(cost IS NULL) AS cost_null, MIN(order_date) AS ngay_dau, MAX(order_date) AS ngay_cuoi
FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`
UNION ALL
SELECT 'fct_net_sales', COUNT(*), COUNT(DISTINCT order_item_id),
       COUNTIF(cost IS NULL), MIN(order_date), MAX(order_date)
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
UNION ALL
SELECT 'staging_complete', COUNT(*), COUNT(DISTINCT order_item_id),
       CAST(NULL AS INT64), MIN(order_date), MAX(order_date)
FROM `bq-learning-510104.dbt_dev.stg_thelook__order_items`
WHERE status = 'Complete'
"""
check(sql)      # xem trước số MB
df = stats(sql) # chạy thật, trần 100 MB
len(df), df

Se quet: 6.61 MB
processed: 6.61 MB | billed: 30.0 MB


(3,
               nguon  so_dong  so_id  cost_null    ngay_dau   ngay_cuoi
 0               inc    45271  45271          0  2019-02-09  2026-10-18
 1     fct_net_sales    45529  45529          0  2019-01-27  2026-10-14
 2  staging_complete    45271  45271       <NA>  2019-02-09  2026-10-18)

In [7]:
# Cell 3
# Mục đích: so id giữa bản chụp cũ (fct_net_sales) và bản mới (fct_net_sales_inc).
# Với id chỉ có ở bản cũ, tra trạng thái hiện tại trong nguồn để biết: đổi trạng thái
# hay biến mất khỏi nguồn.
sql = """
WITH only_old AS (
  SELECT f.order_item_id
  FROM `bq-learning-510104.dbt_dev.fct_net_sales` AS f
  LEFT JOIN `bq-learning-510104.dbt_dev.fct_net_sales_inc` AS i USING (order_item_id)
  WHERE i.order_item_id IS NULL        -- có ở bản cũ, không có ở bản mới
),
only_new AS (
  SELECT i.order_item_id
  FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc` AS i
  LEFT JOIN `bq-learning-510104.dbt_dev.fct_net_sales` AS f USING (order_item_id)
  WHERE f.order_item_id IS NULL        -- có ở bản mới, không có ở bản cũ
)
SELECT 'chi_co_o_ban_cu' AS nhom,
       COALESCE(s.status, '(khong con trong nguon)') AS trang_thai_hien_tai,
       COUNT(*) AS so_dong
FROM only_old AS o
LEFT JOIN `bq-learning-510104.dbt_dev.stg_thelook__order_items` AS s USING (order_item_id)
GROUP BY 1, 2
UNION ALL
SELECT 'chi_co_o_ban_moi', 'Complete', COUNT(*) FROM only_new
ORDER BY nhom, so_dong DESC
"""
check(sql)      # xem trước số MB
df = stats(sql) # chạy thật, trần 100 MB
len(df), df

Se quet: 3.84 MB
processed: 3.84 MB | billed: 30.0 MB


(6,
                nhom      trang_thai_hien_tai  so_dong
 0   chi_co_o_ban_cu                  Shipped    13871
 1   chi_co_o_ban_cu               Processing     8871
 2   chi_co_o_ban_cu                Cancelled     6770
 3   chi_co_o_ban_cu                 Returned     4576
 4   chi_co_o_ban_cu  (khong con trong nguon)      103
 5  chi_co_o_ban_moi                 Complete    33933)

In [8]:
# Cell 4
# Mục đích: sau lần chạy MERGE, kiểm tra 3 điều: không trùng dòng (grain), và bảng incremental
# còn khớp nguồn hiện tại hay đã lệch (thiếu dòng / thừa dòng so với nguồn).
sql = """
SELECT
  (SELECT COUNT(*) FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`) AS so_dong_inc,
  (SELECT COUNT(DISTINCT order_item_id) FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`) AS so_id_inc,
  (SELECT COUNT(*) FROM `bq-learning-510104.dbt_dev.stg_thelook__order_items`
    WHERE status = 'Complete') AS so_dong_nguon,
  -- Complete trong nguồn nhưng chưa có trong bảng inc (thiếu)
  (SELECT COUNT(*)
   FROM `bq-learning-510104.dbt_dev.stg_thelook__order_items` AS s
   LEFT JOIN `bq-learning-510104.dbt_dev.fct_net_sales_inc` AS i USING (order_item_id)
   WHERE s.status = 'Complete' AND i.order_item_id IS NULL) AS thieu_trong_inc,
  -- có trong bảng inc nhưng nguồn không còn là Complete (thừa)
  (SELECT COUNT(*)
   FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc` AS i
   LEFT JOIN `bq-learning-510104.dbt_dev.stg_thelook__order_items` AS s
     ON s.order_item_id = i.order_item_id AND s.status = 'Complete'
   WHERE s.order_item_id IS NULL) AS thua_trong_inc
"""
check(sql)      # xem trước số MB
df = stats(sql) # chạy thật, trần 100 MB
len(df), df

Se quet: 3.5 MB
processed: 3.5 MB | billed: 20.0 MB


(1,
    so_dong_inc  so_id_inc  so_dong_nguon  thieu_trong_inc  thua_trong_inc
 0        45271      45271          45271                0               0)

In [9]:
# Cell 4
# Mục đích: sau lần chạy MERGE, kiểm tra 3 điều: không trùng dòng (grain), và bảng incremental
# còn khớp nguồn hiện tại hay đã lệch (thiếu dòng / thừa dòng so với nguồn).
sql = """
SELECT
  (SELECT COUNT(*) FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`) AS so_dong_inc,
  (SELECT COUNT(DISTINCT order_item_id) FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`) AS so_id_inc,
  (SELECT COUNT(*) FROM `bq-learning-510104.dbt_dev.stg_thelook__order_items`
    WHERE status = 'Complete') AS so_dong_nguon,
  -- Complete trong nguồn nhưng chưa có trong bảng inc (thiếu)
  (SELECT COUNT(*)
   FROM `bq-learning-510104.dbt_dev.stg_thelook__order_items` AS s
   LEFT JOIN `bq-learning-510104.dbt_dev.fct_net_sales_inc` AS i USING (order_item_id)
   WHERE s.status = 'Complete' AND i.order_item_id IS NULL) AS thieu_trong_inc,
  -- có trong bảng inc nhưng nguồn không còn là Complete (thừa)
  (SELECT COUNT(*)
   FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc` AS i
   LEFT JOIN `bq-learning-510104.dbt_dev.stg_thelook__order_items` AS s
     ON s.order_item_id = i.order_item_id AND s.status = 'Complete'
   WHERE s.order_item_id IS NULL) AS thua_trong_inc
"""
check(sql)      # xem trước số MB
df = stats(sql) # chạy thật, trần 100 MB
len(df), df

Se quet: 3.5 MB
processed: 3.5 MB | billed: 20.0 MB


(1,
    so_dong_inc  so_id_inc  so_dong_nguon  thieu_trong_inc  thua_trong_inc
 0        45271      45271          45271                0               0)

In [10]:
# Cell 5
# Mục đích: cố ý xóa các dòng thuộc 2 ngày cuối khỏi fct_net_sales_inc để mô phỏng
# "bảng đích thiếu dữ liệu", rồi dbt run sẽ MERGE thêm lại. Sau Cell này, bảng inc ít hơn nguồn.
sql = """
DELETE FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`
WHERE order_date >= (
  SELECT DATE_SUB(MAX(order_date), INTERVAL 1 DAY)
  FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`
)
"""
# DML (DELETE) không dry run được đầy đủ như SELECT, nên đặt trần byte bằng stats-giống cấu hình
cfg = bigquery.QueryJobConfig(maximum_bytes_billed=100 * 1024 * 1024)
job = client.query(sql, job_config=cfg)
job.result()  # chờ lệnh xong rồi mới đọc số dòng bị xóa
print("So dong da xoa:", job.num_dml_affected_rows)

So dong da xoa: 164


In [11]:
# Cell 5
# Mục đích: đếm trực tiếp số đơn Complete nằm trong cửa sổ 3 ngày cuối của nguồn.
# Nếu bằng 454 (số dòng MERGE ở lần chạy 2) thì chứng minh 454 chỉ là kích thước cửa sổ,
# không phải số dòng đã thay đổi.
sql = """
SELECT
  (SELECT MAX(order_date) FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`) AS ngay_max_trong_inc,
  (SELECT DATE_SUB(MAX(order_date), INTERVAL 3 DAY)
   FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`) AS moc_loc,
  (SELECT COUNT(*)
   FROM `bq-learning-510104.dbt_dev.stg_thelook__order_items`
   WHERE status = 'Complete'
     AND order_date >= (SELECT DATE_SUB(MAX(order_date), INTERVAL 3 DAY)
                        FROM `bq-learning-510104.dbt_dev.fct_net_sales_inc`)) AS so_don_trong_cua_so
"""
check(sql)      # xem trước số MB
df = stats(sql) # chạy thật, trần 100 MB
len(df), df

Se quet: 3.49 MB
processed: 3.49 MB | billed: 20.0 MB


(1,
   ngay_max_trong_inc     moc_loc  so_don_trong_cua_so
 0         2026-10-16  2026-10-13                 1241)